# MSigDB GPU Hypergeometric ORA Analysis: pathway coverage across sample sizes (study subsampling)

**Environment:** `gpu-kmeans` (Python kernel, cupy/RAPIDS)

For each CLAMP model (CLAMPfull and CLAMPbase) across coverage levels, this notebook:

1. Loads the Z matrix (gene loadings per LV) for a given coverage level/seed.
2. Gene universe = all model genes (`Z.index`), matching `clusterProfiler::enricher()`'s convention -- NOT pre-intersected with MSigDB.
3. Runs a one-sided hypergeometric ORA test (`P(X >= x)`, equivalent to Fisher's exact test, alternative="greater") for every (LV, pathway) pair simultaneously via a GPU-vectorized kernel (`libs/gpu_ora.py`): hit set = top 1% of genes per LV by descending loading, pathway DB = MSigDB v2026.1 filtered to size [10, 50000] within the universe.
4. BH-adjusts p-values within each LV, then takes the minimum adjusted p-value per pathway across all LVs (same convention as every sibling ORA/GSEA method in this repo).
5. Saves per-seed CSV caches (`rs{pct}_seed{seed}_msigdb_gpu_ora.csv` + sibling `_meta.csv`) and per-pct-level summary CSVs. FDR thresholds and coverage computation are done in `01_msigdb_gpu_ora_plot.ipynb`.

This track has no native 100% level on disk -- the 100% point reuses the shared full-coverage model from the random-subsampling track (`output/01_model_building/04_archs4/06_bp_coverage_rshall/06_bp_coverage_hall_rs_100`), all 3 seeds.


In [1]:
import os
import time

import pandas as pd

REPO_ROOT = "/home/msubirana/Documents/pivlab/clamp-analyses"
os.chdir(REPO_ROOT)

import sys
sys.path.insert(0, os.path.join(REPO_ROOT, "libs"))
import gpu_ora

t_start = time.time()

## Paths

In [2]:
models_dir = "output/01_model_building/04_archs4/07_bp_coverage_study"
output_dir = "output/03_model_biology/00_archs4/06_coverage_study/gpu_ora_msigdb"

os.makedirs(os.path.join(output_dir, "CLAMPfull"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "CLAMPbase"), exist_ok=True)

## Coverage level specs

Full grid: 1/5/10/25/50/75/100%, seeds 1-3 (all 3 seeds run at every level, including 100%).

In [3]:
coverage_specs = [
    {"pct": 1, "dir": "00_bp_coverage_study_01", "models_dir": models_dir, "prefix": "study_coverage"},
    {"pct": 5, "dir": "01_bp_coverage_study_05", "models_dir": models_dir, "prefix": "study_coverage"},
    {"pct": 10, "dir": "02_bp_coverage_study_10", "models_dir": models_dir, "prefix": "study_coverage"},
    {"pct": 25, "dir": "03_bp_coverage_study_25", "models_dir": models_dir, "prefix": "study_coverage"},
    {"pct": 50, "dir": "04_bp_coverage_study_50", "models_dir": models_dir, "prefix": "study_coverage"},
    {"pct": 75, "dir": "05_bp_coverage_study_75", "models_dir": models_dir, "prefix": "study_coverage"},
    {"pct": 100, "dir": "06_bp_coverage_hall_rs_100", "models_dir": "output/01_model_building/04_archs4/06_bp_coverage_rshall", "prefix": "hall_coverage"},  # shared anchor (no native 100% for this track),
]

seeds = [1, 2, 3]

## Load MSigDB gene sets

In [4]:
library = gpu_ora.read_gmt("data/pathways/msigdb.v2026.1.Hs.symbols.gmt")
print(f"MSigDB gene sets loaded: {len(library)}")

MSigDB gene sets loaded: 35361


## Helper: run GPU ORA across the full grid for one model type

In [5]:
def run_grid(model_subdir, z_subdir):
    results_by_pct = {}

    for spec in coverage_specs:
        pct = spec["pct"]
        pct_csv_path = os.path.join(output_dir, model_subdir, f"results_pct{pct}_msigdb_gpu_ora.csv")

        if os.path.exists(pct_csv_path):
            print(f"Loading cached pct-level result: {model_subdir} {pct}%")
            results_by_pct[pct] = pd.read_csv(pct_csv_path)
            continue

        rows = []
        for s in seeds:
            seed_dir = os.path.join(spec["models_dir"], spec["dir"], f"{spec['prefix']}_rs{pct}_seed_{s}")
            z_path = os.path.join(seed_dir, z_subdir, "Z.csv")
            cache_path = os.path.join(output_dir, model_subdir, f"rs{pct}_seed{s}_msigdb_gpu_ora.csv")
            meta_path = os.path.join(output_dir, model_subdir, f"rs{pct}_seed{s}_meta.csv")

            if not os.path.exists(z_path):
                print(f"SKIP (no Z.csv): {model_subdir} rs{pct} seed{s}")
                continue

            if os.path.exists(cache_path) and os.path.exists(meta_path):
                print(f"Loading cached: {model_subdir} rs{pct} seed{s}")
                meta = pd.read_csv(meta_path).iloc[0]
            else:
                print(f"Running GPU ORA: {model_subdir} rs{pct} seed{s}")
                t0 = time.time()
                res = gpu_ora.run_gpu_ora_for_model(z_path, library, min_size=10, max_size=50000, pct=0.01)
                print(f"  done in {time.time()-t0:.2f}s")
                res["terms_padj"].rename_axis("term").reset_index(name="padj_min_across_lvs").to_csv(cache_path, index=False)
                meta = pd.Series({k: v for k, v in res.items() if k != "terms_padj"})
                meta.to_frame().T.to_csv(meta_path, index=False)

            rows.append({
                "model_type": model_subdir, "coverage_pct": pct, "seed": s,
                "n_samples": meta["n_samples"], "n_lvs": meta["n_lvs"],
                "n_top_genes": meta["n_top_genes"], "n_total_msigdb": meta["n_total_msigdb"],
            })

        if len(rows) == 0:
            print(f"SKIP entire level (no models on disk): {model_subdir} {pct}%")
            continue

        pct_df = pd.DataFrame(rows)
        pct_df.to_csv(pct_csv_path, index=False)
        results_by_pct[pct] = pct_df
        print(f"Saved: {model_subdir} {pct}% -> {pct_csv_path}")

    return pd.concat(results_by_pct.values(), ignore_index=True)

## Run GPU ORA: CLAMPfull

In [6]:
results_full_df = run_grid("CLAMPfull", "CLAMPfull_hall")
print(results_full_df)

Loading cached pct-level result: CLAMPfull 1%
Loading cached pct-level result: CLAMPfull 5%
Loading cached pct-level result: CLAMPfull 10%
Loading cached pct-level result: CLAMPfull 25%
Loading cached pct-level result: CLAMPfull 50%
SKIP (no Z.csv): CLAMPfull rs75 seed1
SKIP (no Z.csv): CLAMPfull rs75 seed2
SKIP (no Z.csv): CLAMPfull rs75 seed3
SKIP entire level (no models on disk): CLAMPfull 75%
Loading cached pct-level result: CLAMPfull 100%
   model_type  coverage_pct  seed  n_samples  n_lvs  n_top_genes  \
0   CLAMPfull             1     1       6109    134          185   
1   CLAMPfull             1     2       7084    186          185   
2   CLAMPfull             1     3       7858    212          185   
3   CLAMPfull             5     1      30308    686          185   
4   CLAMPfull             5     2      30290    842          185   
5   CLAMPfull             5     3      30281    752          185   
6   CLAMPfull            10     1      60645    944          185   
7   CLAM

## Run GPU ORA: CLAMPbase

In [7]:
results_base_df = run_grid("CLAMPbase", "CLAMPbase")
print(results_base_df)

Loading cached pct-level result: CLAMPbase 1%
Loading cached pct-level result: CLAMPbase 5%
Loading cached pct-level result: CLAMPbase 10%
Loading cached pct-level result: CLAMPbase 25%
Loading cached pct-level result: CLAMPbase 50%
SKIP (no Z.csv): CLAMPbase rs75 seed1
SKIP (no Z.csv): CLAMPbase rs75 seed2
SKIP (no Z.csv): CLAMPbase rs75 seed3
SKIP entire level (no models on disk): CLAMPbase 75%
Loading cached pct-level result: CLAMPbase 100%
   model_type  coverage_pct  seed  n_samples  n_lvs  n_top_genes  \
0   CLAMPbase             1     1       6109    134          185   
1   CLAMPbase             1     2       7084    186          185   
2   CLAMPbase             1     3       7858    212          185   
3   CLAMPbase             5     1      30308    686          185   
4   CLAMPbase             5     2      30290    842          185   
5   CLAMPbase             5     3      30281    752          185   
6   CLAMPbase            10     1      60645    944          185   
7   CLAM

In [8]:
print(f"Total notebook time: {(time.time()-t_start)/60:.1f} min")

Total notebook time: 0.0 min
